# HAM10000 — Textual Inversion

Aprende el token `<mel-skin>` a partir de las ~800 imágenes de melanoma reales del split de entrenamiento.

Resultado: un vector de embedding que captura la apariencia visual del melanoma dermoscópico
y puede usarse como condicionamiento en Stable Diffusion.

| Hardware | Steps | Tiempo estimado |
|---|---|---|
| Google Colab T4 | 5 000 | ~30 min |
| Apple Silicon MPS | 5 000 | ~2–4 h |
| CPU | 5 000 | ~6–12 h |

Para una prueba rápida cambia `TI_STEPS = 200` en la celda de hiperparámetros.

**Prerequisito**: `data/processed/melanoma_train_for_colab.zip` (local)  
o ZIP subido a Google Drive `Mi unidad/ham10000-augmentation/melanoma_train_for_colab.zip` (Colab)

In [1]:
# Detectar entorno: Colab vs local
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Instalar diffusers si no está disponible
try:
    import diffusers
    print(f'diffusers {diffusers.__version__}')
except ImportError:
    import subprocess, sys
    print('Instalando dependencias (primera vez ~1 min)...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '-U',
        'diffusers', 'transformers', 'accelerate', 'safetensors', 'huggingface_hub',
    ])
    import diffusers

import torch

def get_device():
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'CUDA GPU: {name}  ({vram:.1f} GB)')
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        print('Apple Silicon MPS')
        return torch.device('mps')
    print('Sin GPU — usando CPU (entrenamiento muy lento)')
    return torch.device('cpu')

DEVICE = get_device()
DTYPE  = torch.float16 if DEVICE.type == 'cuda' else torch.float32
print(f'device={DEVICE}  dtype={DTYPE}')

diffusers 0.37.1
CUDA GPU: Tesla T4  (15.6 GB)
device=cuda  dtype=torch.float16


In [2]:
import zipfile
from pathlib import Path

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/ham10000-augmentation')
    ZIP_PATH   = DRIVE_ROOT / 'melanoma_train_for_colab.zip'
    IMAGES_DIR = Path('/content/melanoma_images')
    MODEL_DIR  = DRIVE_ROOT / 'models'
else:
    # Lanza Jupyter desde la raiz del proyecto o ajusta PROJECT_ROOT
    PROJECT_ROOT = Path.cwd()
    ZIP_PATH     = PROJECT_ROOT / 'data/processed/melanoma_train_for_colab.zip'
    IMAGES_DIR   = PROJECT_ROOT / 'data/processed/melanoma_images_tmp'
    MODEL_DIR    = PROJECT_ROOT / 'data/processed/ti_models'

for d in (IMAGES_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.exists(), (
    f'ZIP no encontrado:\n  {ZIP_PATH}\n\n'
    'Ejecuta: python scripts/augmentation/prepare_for_colab.py'
)
print(f'ZIP:      {ZIP_PATH}')
print(f'Imagenes: {IMAGES_DIR}')
print(f'Modelos:  {MODEL_DIR}')

Mounted at /content/drive
ZIP:      /content/drive/MyDrive/ham10000-augmentation/melanoma_train_for_colab.zip
Imagenes: /content/melanoma_images
Modelos:  /content/drive/MyDrive/ham10000-augmentation/models


In [3]:
if not any(IMAGES_DIR.glob('*.jpg')):
    print('Extrayendo imagenes del ZIP...')
    with zipfile.ZipFile(ZIP_PATH) as zf:
        for m in zf.infolist():
            if m.filename.startswith('images/') and m.filename.endswith('.jpg'):
                data = zf.read(m.filename)
                (IMAGES_DIR / Path(m.filename).name).write_bytes(data)
    print('Listo')
else:
    print('Imagenes ya extraidas, saltando extraccion')

image_paths = sorted(IMAGES_DIR.glob('*.jpg'))
print(f'Imagenes disponibles: {len(image_paths)}')

Extrayendo imagenes del ZIP...
Listo
Imagenes disponibles: 801


## Hiperparámetros

Cambia `TI_STEPS = 200` para una prueba rápida (~2 min en T4).

In [4]:
MODEL_ID      = 'runwayml/stable-diffusion-v1-5'
PLACEHOLDER   = '<mel-skin>'
INIT_TOKEN    = 'melanoma'   # token que inicializa el nuevo embedding
PROMPT_TMPL   = 'a dermoscopic image of a {} skin lesion'

TI_STEPS      = 5000        # reducir a 200 para prueba rapida
TI_LR         = 5e-4
TI_SAVE_EVERY = 500         # guardar checkpoint cada N steps
RESOLUTION    = 512

In [5]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class MelanomaDataset(Dataset):
    def __init__(self, paths, placeholder, tokenizer, resolution=512):
        self.paths       = paths
        self.placeholder = placeholder
        self.tokenizer   = tokenizer
        self.tf          = transforms.Compose([
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(resolution),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img       = Image.open(self.paths[idx % len(self.paths)]).convert('RGB')
        pv        = self.tf(img)
        prompt    = PROMPT_TMPL.format(self.placeholder)
        input_ids = self.tokenizer(
            prompt, padding='max_length', truncation=True,
            max_length=self.tokenizer.model_max_length, return_tensors='pt'
        ).input_ids[0]
        return {'pixel_values': pv, 'input_ids': input_ids}

In [6]:
from transformers import CLIPTokenizer, CLIPTextModel
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler

print('Cargando modelos (primera vez descarga ~4 GB)...')
tokenizer       = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder='tokenizer')
text_encoder    = CLIPTextModel.from_pretrained(MODEL_ID, subfolder='text_encoder').to(DEVICE)
vae             = AutoencoderKL.from_pretrained(MODEL_ID, subfolder='vae').to(DEVICE, dtype=DTYPE)
unet            = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder='unet').to(DEVICE, dtype=DTYPE)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder='scheduler')
print('Modelos cargados')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Cargando modelos (primera vez descarga ~4 GB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

Modelos cargados


In [7]:
# Registrar token placeholder (idempotente: no falla si ya existe)
if PLACEHOLDER not in tokenizer.get_vocab():
    tokenizer.add_tokens([PLACEHOLDER])
    print(f"Token '{PLACEHOLDER}' anadido al vocabulario")
else:
    print(f"Token '{PLACEHOLDER}' ya estaba en el vocabulario")

placeholder_id = tokenizer.convert_tokens_to_ids(PLACEHOLDER)
text_encoder.resize_token_embeddings(len(tokenizer))

assert text_encoder.get_input_embeddings().weight.shape[0] == len(tokenizer), \
    'Inconsistencia entre vocabulario y embedding'

# Inicializar el nuevo embedding con INIT_TOKEN
init_id = tokenizer.encode(INIT_TOKEN, add_special_tokens=False)[0]
with torch.no_grad():
    w = text_encoder.get_input_embeddings().weight
    w[placeholder_id] = w[init_id].clone()

# Congelar todo excepto el embedding del nuevo token
vae.requires_grad_(False)
unet.requires_grad_(False)
text_encoder.requires_grad_(False)
text_encoder.get_input_embeddings().weight.requires_grad_(True)

optimizer = torch.optim.AdamW(
    [text_encoder.get_input_embeddings().weight],
    lr=TI_LR, betas=(0.9, 0.999), eps=1e-8,
)
print(f"'{PLACEHOLDER}'  ID={placeholder_id}  vocab={len(tokenizer)}  emb={w.shape[0]}")

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Token '<mel-skin>' anadido al vocabulario
'<mel-skin>'  ID=49408  vocab=49409  emb=49409


In [8]:
import torch.nn.functional as F
from tqdm.auto import tqdm

# pin_memory y num_workers solo tienen sentido con CUDA
pin      = DEVICE.type == 'cuda'
n_workers = 2 if DEVICE.type == 'cuda' else 0
ds       = MelanomaDataset(image_paths, PLACEHOLDER, tokenizer, RESOLUTION)
loader   = DataLoader(ds, batch_size=1, shuffle=True, num_workers=n_workers, pin_memory=pin)

data_iter = iter(loader)
text_encoder.train()
losses    = []
pbar      = tqdm(range(1, TI_STEPS + 1), desc='Textual Inversion')

for step in pbar:
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        batch     = next(data_iter)

    pixel_values = batch['pixel_values'].to(DEVICE, dtype=DTYPE)
    input_ids    = batch['input_ids'].to(DEVICE)

    with torch.no_grad():
        latents = vae.encode(pixel_values).latent_dist.sample() * 0.18215

    noise     = torch.randn_like(latents)
    timesteps = torch.randint(
        0, noise_scheduler.config.num_train_timesteps, (1,), device=DEVICE
    ).long()
    noisy_lat = noise_scheduler.add_noise(latents, noise, timesteps)

    encoder_hidden = text_encoder(input_ids)[0].to(dtype=DTYPE)
    noise_pred     = unet(noisy_lat, timesteps, encoder_hidden).sample
    loss           = F.mse_loss(noise_pred.float(), noise.float())

    loss.backward()

    # Solo actualizar el gradiente del token nuevo; congelar el resto
    with torch.no_grad():
        g    = text_encoder.get_input_embeddings().weight.grad
        mask = torch.zeros_like(g)
        mask[placeholder_id] = 1.0
        g.mul_(mask)

    optimizer.step()
    optimizer.zero_grad()

    losses.append(loss.item())
    avg50 = sum(losses[-50:]) / min(len(losses), 50)
    pbar.set_postfix(loss=f'{loss.item():.4f}', avg50=f'{avg50:.4f}')

    if step % TI_SAVE_EVERY == 0:
        emb_ckpt = text_encoder.get_input_embeddings().weight[placeholder_id].detach().cpu()
        torch.save(
            {'embedding': emb_ckpt, 'step': step, 'token': PLACEHOLDER},
            MODEL_DIR / f'mel_skin_embedding_step{step}.pt',
        )
        print(f'  Checkpoint guardado: step {step}')

print('\nEntrenamiento TI completado')

Textual Inversion:   0%|          | 0/5000 [00:00<?, ?it/s]

  Checkpoint guardado: step 500
  Checkpoint guardado: step 1000
  Checkpoint guardado: step 1500
  Checkpoint guardado: step 2000
  Checkpoint guardado: step 2500
  Checkpoint guardado: step 3000
  Checkpoint guardado: step 3500
  Checkpoint guardado: step 4000
  Checkpoint guardado: step 4500
  Checkpoint guardado: step 5000

Entrenamiento TI completado


In [9]:
final_emb = text_encoder.get_input_embeddings().weight[placeholder_id].detach().cpu()
out_path  = MODEL_DIR / 'mel_skin_embedding_final.pt'
torch.save({
    'embedding':  final_emb,
    'step':       TI_STEPS,
    'token':      PLACEHOLDER,
    'init_token': INIT_TOKEN,
    'model_id':   MODEL_ID,
}, out_path)
print(f'Embedding guardado:\n  {out_path}')
print(f'\nSiguiente paso -> HAM10000_generation.ipynb')

Embedding guardado:
  /content/drive/MyDrive/ham10000-augmentation/models/mel_skin_embedding_final.pt

Siguiente paso -> HAM10000_generation.ipynb


## Siguientes pasos

1. Verifica que `mel_skin_embedding_final.pt` se guardó correctamente.
2. Si usas Colab, el archivo está en Google Drive bajo `models/`.
3. Abre `HAM10000_generation.ipynb` para generar las imágenes sintéticas.